# Homework 06: Inverse Dynamics & Static Optimization**Computational Sensorimotor Control — Week 6****Due:** Before Week 7 lab**Task context:** All questions use the same 6 cm semicircular arc from the lecture and lab, starting from Q_REF = (55°, 75°), under a 3 N load in −y.**Part 1** examines how movement speed affects the gap between EPH and inverse dynamics.**Part 2** examines how the choice of optimization criterion shapes the predicted muscle activation pattern.

In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize as scipy_minimize
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

from smc import (
    Arm, make_muscles, simulate_lambda,
    Q_REF, MUSCLE_DEFS,
    mass_matrix, coriolis, joint_accelerations, rk4_step,
)

arm = Arm()
ik  = arm.inverse_kinematics
fk  = arm.forward_kinematics
start_hand = fk(Q_REF)

R_ARC = 0.06
F_LOAD = np.array([0.0, -3.0])
J_ref = arm.jacobian(Q_REF)
tau_ext = J_ref.T @ F_LOAD
perturb_3N = lambda t: tau_ext

NAVY='#1B2A4A'; TEAL='#2E86AB'; GREEN='#27AE60'; RED='#E74C3C'; ORANGE='#E67E22'; PURPLE='#8E44AD'

# ── Shared helpers (same as lab) ──

def arc_geometry(R):
    center = start_hand + np.array([R, 0])
    tip    = start_hand + np.array([R, R])
    end    = start_hand + np.array([2*R, 0])
    return center, tip, end

def dense_arc(R, n=500):
    center = start_hand + np.array([R, 0])
    th = np.linspace(np.pi, 0, n)
    return np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def max_path_deviation(hand, desired):
    return max(np.linalg.norm(desired - hand[i], axis=1).min() for i in range(len(hand)))

def lambda_for_posture(q_target, C=0.25):
    muscles = make_muscles()
    lam = np.zeros(6)
    for j, m in enumerate(muscles):
        l_eq = m.length(q_target)
        shift = (abs(m.r_sh) + abs(m.r_el)) * C
        lam[j] = l_eq - shift
    return lam

def minjerk_arc(R, T, dt=0.001):
    center = start_hand + np.array([R, 0])
    t = np.arange(0, T + dt, dt)
    tau = t / T
    s = 10*tau**3 - 15*tau**4 + 6*tau**5
    theta = np.pi * (1 - s)
    pos = np.column_stack([center[0]+R*np.cos(theta), center[1]+R*np.sin(theta)])
    return t, pos

def eph_arc(R, T_move, dy=0, C=0.25, perturb_fn=None, dt=0.0005):
    _, tip, end = arc_geometry(R)
    tip_s = tip + np.array([0, dy]); end_s = end + np.array([0, dy])
    try:
        q_tip = ik(tip_s[0], tip_s[1]); q_end = ik(end_s[0], end_s[1])
        if not arm.in_joint_limits(q_tip) or not arm.in_joint_limits(q_end): return None, None, None
    except: return None, None, None
    li = lambda_for_posture(Q_REF, C)
    l_tip = lambda_for_posture(q_tip, C); l_end = lambda_for_posture(q_end, C)
    ramp = T_move / 2
    def lam_fn(t):
        if t < ramp: return li + (t/ramp)*(l_tip - li)
        elif t < 2*ramp: return l_tip + ((t-ramp)/ramp)*(l_end - l_tip)
        else: return l_end.copy()
    t_arr, states, hand, acts = simulate_lambda(lam_fn, T=T_move+0.4, dt=dt, perturb_fn=perturb_fn)
    return t_arr, hand, acts

def eph_continuous(R, T_move, dy=0, C=0.25, perturb_fn=None, dt=0.0005):
    t_mj, pos_mj = minjerk_arc(R, T_move, dt=dt)
    pos_shifted = pos_mj.copy(); pos_shifted[:, 1] += dy
    n_mj = len(t_mj)
    for k in range(0, n_mj, max(1, n_mj//20)):
        try:
            q = ik(pos_shifted[k,0], pos_shifted[k,1])
            if not arm.in_joint_limits(q): return None
        except: return None
    def lam_fn(t):
        if t >= T_move: q_d = ik(pos_shifted[-1,0], pos_shifted[-1,1])
        else:
            idx = min(int(t/dt), n_mj-1)
            q_d = ik(pos_shifted[idx,0], pos_shifted[idx,1])
        return lambda_for_posture(q_d, C)
    _, _, h, _ = simulate_lambda(lam_fn, T=T_move+0.4, dt=dt, perturb_fn=perturb_fn)
    return h

def grid_search_eph(R, T_move, mode='discrete', perturb_fn=perturb_3N):
    des = dense_arc(R)
    best = 999; bdy = 0; bC = 0.25
    for dy in np.linspace(0, 0.08, 12):
        for C in np.linspace(0.20, 0.60, 10):
            if mode == 'discrete':
                _, h, _ = eph_arc(R, T_move, dy=dy, C=C, perturb_fn=perturb_fn)
            else:
                h = eph_continuous(R, T_move, dy=dy, C=C, perturb_fn=perturb_fn)
            if h is None: continue
            v = max_path_deviation(h, des)
            if v < best: best = v; bdy = dy; bC = C
    return bdy, bC, best

def run_id(R, T_move, F_ext=None, dt=0.001):
    t_mj, pos = minjerk_arc(R, T_move, dt=dt)
    n = len(t_mj)
    q = np.zeros((n,2))
    for i in range(n): q[i] = ik(pos[i,0], pos[i,1])
    qd = np.zeros_like(q); qdd = np.zeros_like(q)
    qd[1:-1] = (q[2:]-q[:-2])/(2*dt); qd[0]=qd[1]; qd[-1]=qd[-2]
    qdd[1:-1] = (q[2:]-2*q[1:-1]+q[:-2])/dt**2; qdd[0]=qdd[1]; qdd[-1]=qdd[-2]
    tau = np.zeros((n,2)); tau_M = np.zeros((n,2)); tau_C = np.zeros((n,2))
    for i in range(n):
        M = mass_matrix(q[i]); C = coriolis(q[i], qd[i])
        tau_M[i] = M @ qdd[i]; tau_C[i] = C
        tau[i] = tau_M[i] + C
        if F_ext is not None: tau[i] -= arm.jacobian(q[i]).T @ F_ext
    st = np.concatenate([Q_REF.copy(),[0.,0.]]); hand = np.zeros((n,2))
    for i in range(n):
        hand[i] = fk(st[:2])
        t_s = tau[i] + (arm.jacobian(st[:2]).T @ F_ext if F_ext is not None else 0)
        def d(s, _t=t_s):
            a = joint_accelerations(s[:2],s[2:],_t)
            return np.array([s[2],s[3],a[0],a[1]])
        st = rk4_step(st, d, dt)
    return t_mj, hand, tau, tau_M, tau_C, q, qd, qdd

des6 = dense_arc(R_ARC)
_, tip6, end6 = arc_geometry(R_ARC)
print('Setup complete.')

---## Part 1: Movement Speed and the Dynamics Gap (6 questions)In the lab, we used a fixed movement time of 800 ms. Here you will test two different speeds — a faster movement (600 ms) and a slower movement (1000 ms) — to understand how speed affects the gap between EPH and inverse dynamics.**Reminder:** The grid search finds EPH's theoretical best — the brain would learn these parameters from experience, not compute them on the fly.

### Q1.1: Run all controllers at 600 msFor T = 600 ms on the 6 cm arc under 3 N load, compute path deviation for:- Discrete EPH (optimized via grid search)- Continuous EPH (optimized via grid search)- ID + min-jerk**Your tasks:** Fill in the TODO lines to run each controller at the faster speed.

In [ ]:
T_FAST = 0.600

# Grid search for discrete EPH at 600 ms
# TODO 1: Run grid_search_eph for discrete mode at T_FAST
# ← YOUR CODE HERE

# Grid search for continuous EPH at 600 ms
# TODO 2: Run grid_search_eph for continuous mode at T_FAST
# ← YOUR CODE HERE

# ID + min-jerk at 600 ms
_, hand_id_fast, _, _, _, _, _, _ = run_id(R_ARC, T_FAST, F_ext=F_LOAD)
pd_id_fast = max_path_deviation(hand_id_fast, des6)

print(f'T = {T_FAST*1000:.0f} ms:')
print(f'  Discrete EPH opt:   dy={dy_d_fast*100:.1f}cm, C={C_d_fast:.2f}, pd={pd_d_fast*100:.2f} cm')
print(f'  Continuous EPH opt: dy={dy_c_fast*100:.1f}cm, C={C_c_fast:.2f}, pd={pd_c_fast*100:.2f} cm')
print(f'  ID + min-jerk:      pd={pd_id_fast*100:.4f} cm')

### Q1.2: Run all controllers at 1000 msSame analysis at the slower speed.**Your tasks:** Fill in the TODO lines.

In [ ]:
T_SLOW = 1.000

# TODO 1: Run grid_search_eph for discrete mode at T_SLOW
# ← YOUR CODE HERE

# TODO 2: Run grid_search_eph for continuous mode at T_SLOW
# ← YOUR CODE HERE

# ID + min-jerk at 1000 ms
_, hand_id_slow, _, _, _, _, _, _ = run_id(R_ARC, T_SLOW, F_ext=F_LOAD)
pd_id_slow = max_path_deviation(hand_id_slow, des6)

print(f'T = {T_SLOW*1000:.0f} ms:')
print(f'  Discrete EPH opt:   dy={dy_d_slow*100:.1f}cm, C={C_d_slow:.2f}, pd={pd_d_slow*100:.2f} cm')
print(f'  Continuous EPH opt: dy={dy_c_slow*100:.1f}cm, C={C_c_slow:.2f}, pd={pd_c_slow*100:.2f} cm')
print(f'  ID + min-jerk:      pd={pd_id_slow*100:.4f} cm')

### Q1.3: Summary tableCombine results from Q1.1, Q1.2, and the 800 ms lab results into a single comparison.

In [ ]:
# Run 800 ms for completeness (same as lab)
dy_d_800, C_d_800, pd_d_800 = grid_search_eph(R_ARC, 0.800, mode='discrete')
dy_c_800, C_c_800, pd_c_800 = grid_search_eph(R_ARC, 0.800, mode='continuous')
_, hand_id_800, _, _, _, _, _, _ = run_id(R_ARC, 0.800, F_ext=F_LOAD)
pd_id_800 = max_path_deviation(hand_id_800, des6)

print(f'{"T (ms)":>7} | {"Disc EPH":>10} {"Cont EPH":>10} {"ID+MJ":>10} | {"Disc/ID":>8} {"Cont/ID":>8}')
print('-' * 68)
for T_ms, pd_d, pd_c, pd_i in [
    (600, pd_d_fast*100, pd_c_fast*100, pd_id_fast*100),
    (800, pd_d_800*100, pd_c_800*100, pd_id_800*100),
    (1000, pd_d_slow*100, pd_c_slow*100, pd_id_slow*100),
]:
    print(f'{T_ms:6d}  | {pd_d:9.2f} {pd_c:9.2f} {pd_i:9.4f} | {pd_d/pd_i:7.0f}x {pd_c/pd_i:7.0f}x')

### Q1.4: Torque decomposition at both speedsPlot the shoulder torque decomposition (Mq̈ vs C) at 600 ms and 1000 ms to see which term drives the speed effect.**Your task:** Fill in the TODO line to run the ID pipeline at the slow speed.

In [ ]:
# Fast (600 ms)
t_fast, _, tau_fast, tauM_fast, tauC_fast, _, _, _ = run_id(R_ARC, T_FAST, F_ext=F_LOAD)

# TODO 1: Run ID at T_SLOW to get the torque decomposition
# ← YOUR CODE HERE

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(t_fast*1000, tauM_fast[:,0], TEAL, lw=2, label='Mq\u0308 (inertial)')
ax1.plot(t_fast*1000, tauC_fast[:,0], ORANGE, lw=2, label='C (Coriolis)')
ax1.set_xlabel('Time (ms)'); ax1.set_ylabel('Torque (N\u00B7m)')
ax1.set_title(f'A. Shoulder torque, T = {T_FAST*1000:.0f} ms'); ax1.legend()

ax2.plot(t_slow*1000, tauM_slow[:,0], TEAL, lw=2, label='Mq\u0308 (inertial)')
ax2.plot(t_slow*1000, tauC_slow[:,0], ORANGE, lw=2, label='C (Coriolis)')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Torque (N\u00B7m)')
ax2.set_title(f'B. Shoulder torque, T = {T_SLOW*1000:.0f} ms'); ax2.legend()

plt.tight_layout(); plt.show()

print(f'Peak |Mq\u0308| at 600 ms: {np.abs(tauM_fast[:,0]).max():.2f} N\u00B7m')
print(f'Peak |Mq\u0308| at 1000 ms: {np.abs(tauM_slow[:,0]).max():.2f} N\u00B7m')
print(f'Ratio: {np.abs(tauM_fast[:,0]).max() / np.abs(tauM_slow[:,0]).max():.1f}x')

### Q1.5: Written answer (2–3 sentences)Based on your summary table (Q1.3) and torque plots (Q1.4):**Does the gap between EPH and ID grow or shrink at faster speeds? Which specific term in the inverse dynamics equation (Mq̈, C, or J^T·F) is primarily responsible? Explain why EPH's spring-like forces cannot compensate for this term.**

In [ ]:
# Q1.5: Write your answer here as a comment or markdown cell.
#
# Write your answer here.


### Q1.6: Written answer (1–2 sentences)**The optimal C-command from the grid search may differ between 600 ms and 1000 ms. If it does, explain why. If it doesn't, explain why not.**

In [ ]:
# Q1.6: Write your answer here.
#
# Write your answer here.


---## Part 2: The Crowninshield Exponent (6 questions)In the lab (Part 5), we used n = 3 for the Crowninshield criterion: minimize Σ(Fᵢ/ρᵢ)ⁿ. Here you explore what happens when n changes — and which n produces muscle activations most similar to EPH.**Reading connection:** Crowninshield & Brand (1981), §3, discuss why they preferred n = 3 over n = 2.

### Q2.1: Static optimization at five exponentsRun static optimization on the 6 cm arc (3 N load, 800 ms) for n = 1, 2, 3, 4, 5 and store the full-trajectory forces.**Your task:** Fill in the TODO line to call the optimizer with the current exponent.

In [ ]:
# ID torques for 6 cm arc, 800 ms, 3 N load
T_HW = 0.800; dt_hw = 0.001
t_hw, pos_hw = minjerk_arc(R_ARC, T_HW, dt=dt_hw)
n_hw = len(t_hw)

q_hw = np.zeros((n_hw, 2))
for i in range(n_hw): q_hw[i] = ik(pos_hw[i,0], pos_hw[i,1])
qd_hw = np.zeros_like(q_hw); qdd_hw = np.zeros_like(q_hw)
qd_hw[1:-1] = (q_hw[2:]-q_hw[:-2])/(2*dt_hw); qd_hw[0]=qd_hw[1]; qd_hw[-1]=qd_hw[-2]
qdd_hw[1:-1] = (q_hw[2:]-2*q_hw[1:-1]+q_hw[:-2])/dt_hw**2; qdd_hw[0]=qdd_hw[1]; qdd_hw[-1]=qdd_hw[-2]

tau_hw = np.zeros((n_hw, 2))
for i in range(n_hw):
    M = mass_matrix(q_hw[i]); C = coriolis(q_hw[i], qd_hw[i])
    J = arm.jacobian(q_hw[i])
    tau_hw[i] = M @ qdd_hw[i] + C - J.T @ F_LOAD

# Moment arms and ρ
R_mat = np.array([[m[3] for m in MUSCLE_DEFS], [m[4] for m in MUSCLE_DEFS]])
rho = np.array([m[1] for m in MUSCLE_DEFS])
mnames = ['Pec','BicL','BicS','Delt','TriL','TriLg']
mcols = [RED, ORANGE, '#E8735A', TEAL, '#3498db', GREEN]

def static_opt(tau_req, n_pow):
    def cost(F):
        return np.sum((F / rho) ** n_pow)
    def cost_jac(F):
        return n_pow * (F / rho) ** (n_pow - 1) / rho
    constraints = {'type': 'eq', 'fun': lambda F: R_mat @ F - tau_req, 'jac': lambda F: R_mat}
    res = scipy_minimize(cost, np.ones(6)*0.1, jac=cost_jac, method='SLSQP',
                         bounds=[(0,None)]*6, constraints=constraints,
                         options={'maxiter': 500, 'ftol': 1e-12})
    return res.x if res.success else np.zeros(6)

# Run optimization for n = 1, 2, 3, 4, 5
exponents = [1, 2, 3, 4, 5]
F_by_n = {}

for n_pow in exponents:
    F_all = np.zeros((n_hw, 6))
    for i in range(n_hw):
        if np.linalg.norm(tau_hw[i]) > 1e-6:
            # TODO 1: Call static_opt with the current torque and exponent
            # ← YOUR CODE HERE
    F_by_n[n_pow] = F_all
    print(f'n={n_pow} done — peak forces: {[f"{F_all[:,j].max():.1f}" for j in range(6)]}')

### Q2.2: Plot muscle forces for n = 1, 3, 5Show three panels side by side: the full-trajectory muscle forces for n = 1 (sparse), n = 3 (Crowninshield), and n = 5 (heavily distributed).**Your task:** Fill in the TODO line to plot the forces for each exponent.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for idx, n_pow in enumerate([1, 3, 5]):
    ax = axes[idx]
    for j in range(6):
        # TODO 1: Plot the force trajectory for muscle j at this exponent
        # ← YOUR CODE HERE
    ax.set_xlabel('Time (ms)'); ax.set_ylabel('Force (N)')
    ax.set_title(f'n = {n_pow}', fontweight='bold')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('Static optimization: effect of exponent n (6 cm arc, 3 N load)', fontweight='bold')
plt.tight_layout(); plt.show()

### Q2.3: Count active muscles at peak torqueFor each exponent, count how many muscles are "active" (force > 1 N) at the peak-torque timestep. This shows how n controls the sparsity of the solution.**Your task:** Fill in the TODO line to count active muscles.

In [ ]:
i_peak = np.argmax(np.linalg.norm(tau_hw, axis=1))

print(f'Peak torque at t = {t_hw[i_peak]*1000:.0f} ms')
print(f'\u03C4 = ({tau_hw[i_peak, 0]:.2f}, {tau_hw[i_peak, 1]:.2f}) N\u00B7m')
print()
print(f'{"n":>3} | {"Active muscles":>15} | Forces')
print('-' * 60)

for n_pow in exponents:
    forces = F_by_n[n_pow][i_peak]
    # TODO 1: Count how many muscles have force > 1 N
    # ← YOUR CODE HERE
    active_list = ', '.join(f'{mnames[j]}={forces[j]:.1f}' for j in range(6) if forces[j] > 1.0)
    print(f'{n_pow:3d} | {n_active:15d} | {active_list}')

### Q2.4: Compare ID activations against EPHCompute a similarity metric between the normalized ID activation pattern (for each n) and the normalized EPH activation pattern. Use the mean correlation across all 6 muscles.**Your task:** Fill in the TODO line to compute the Pearson correlation.

In [ ]:
# Get EPH activations (same as lab)
t_eph, _, a_eph = eph_arc(R_ARC, T_HW, dy=0, C=0.25, perturb_fn=perturb_3N, dt=0.0002)

# Resample to 1 ms
a_eph_1ms = np.zeros((n_hw, 6))
for j in range(6):
    a_eph_1ms[:, j] = np.interp(t_hw, t_eph, a_eph[:, j])

# Normalize to [0, 1]
def norm01(arr):
    out = np.zeros_like(arr)
    for j in range(arr.shape[1]):
        mx = arr[:, j].max()
        if mx > 0: out[:, j] = arr[:, j] / mx
    return out

a_eph_norm = norm01(a_eph_1ms)

print(f'{"n":>3} | {"Mean corr":>10} | Per-muscle correlations')
print('-' * 70)

best_n = None; best_corr = -1

for n_pow in exponents:
    a_id_norm = norm01(F_by_n[n_pow])
    corrs = []
    for j in range(6):
        if a_id_norm[:, j].std() > 1e-6 and a_eph_norm[:, j].std() > 1e-6:
            # TODO 1: Compute Pearson correlation between ID and EPH for muscle j
            # ← YOUR CODE HERE
            corrs.append(r)
        else:
            corrs.append(0.0)
    mean_corr = np.mean(corrs)
    if mean_corr > best_corr:
        best_corr = mean_corr; best_n = n_pow
    corr_str = ', '.join(f'{mnames[j]}={corrs[j]:.2f}' for j in range(6))
    print(f'{n_pow:3d} | {mean_corr:10.3f} | {corr_str}')

print(f'\nMost similar to EPH: n = {best_n} (mean r = {best_corr:.3f})')

### Q2.5: Co-contraction comparisonCompute the total co-contraction for each exponent and compare against EPH.**Your task:** Fill in the TODO line to compute co-contraction for each exponent.

In [ ]:
def cocontraction_total(F_arr):
    """Sum of min(agonist, antagonist) torque at each joint, integrated over time."""
    cc_total = 0
    for i in range(len(F_arr)):
        sh_flex = F_arr[i,0]*abs(R_mat[0,0]) + F_arr[i,2]*abs(R_mat[0,2])
        sh_ext  = F_arr[i,3]*abs(R_mat[0,3]) + F_arr[i,5]*abs(R_mat[0,5])
        el_flex = F_arr[i,1]*abs(R_mat[1,1]) + F_arr[i,2]*abs(R_mat[1,2])
        el_ext  = F_arr[i,4]*abs(R_mat[1,4]) + F_arr[i,5]*abs(R_mat[1,5])
        cc_total += min(sh_flex, sh_ext) + min(el_flex, el_ext)
    return cc_total * dt_hw  # integrate over time

print(f'{"n":>3} | {"Total co-contraction":>20}')
print('-' * 30)
for n_pow in exponents:
    # TODO 1: Compute total co-contraction for this exponent
    # ← YOUR CODE HERE
    print(f'{n_pow:3d} | {cc:20.4f}')

cc_eph = cocontraction_total(a_eph_1ms)
print(f'\nEPH total co-contraction: {cc_eph:.4f}')
print('(Note: EPH units are threshold displacement, not Newtons — the absolute')
print(' values are not directly comparable, but the relative pattern is informative.)')

### Q2.6: Written answer (3–4 sentences)Based on your results from Q2.1–Q2.5 and the Crowninshield & Brand (1981) reading:**a) Which value of n makes the ID activation pattern most similar to EPH? Why do you think this is the case?****b) Crowninshield & Brand preferred n = 3. Based on your co-contraction results and the number of active muscles, do you think n = 3 is a good model for what the brain does? Why or why not?**

In [ ]:
# Q2.6: Write your answer here.
#
# Write your answer here.
